In [2]:
import torch
import os
import json
import numpy as np
import pandas as pd
import random
import statistics
import matplotlib.pyplot as plt
import matplotlib
from risk_control_utils import (get_label_order, rc_main, get_all_confidences, get_all_accuracies, get_losses_and_exits_confidence,
                                    get_ground_truth_by_type, get_relative_labels, apply_risk_control, load_all_data)

In [3]:
# Define a list of all the models and datasets to plot
models = ["facebook/layerskip-llama3-8B", "facebook/layerskip-llama2-7B", "meta-llama/Meta-Llama-3-8B", "meta-llama/Llama-2-7B-hf", 
          'meta-llama/Llama-2-13B-hf']
tokenizers = ["meta-llama/Meta-Llama-3-8B", "meta-llama/Llama-2-7B-hf", "meta-llama/Meta-Llama-3-8B", "meta-llama/Llama-2-7B-hf", 
              'meta-llama/Llama-2-13B-hf']
n_early_exits = [32, 32, 32, 32, 40]
datasets = ['financial_phrasebank', 'sst2', 'tweeteval_hate', 'tweeteval_feminist', 'tweeteval_atheism', 'unnatural',
            'boolean', 'navigation', 'sports', 'web_of_lies']
# datasets = ['boolean', 'navigation', 'sports', 'web_of_lies']

In [4]:
# Define base data folder to read from (and associated params for finding the right data files)
results_folder = './results_icl_rc_data_no_calibration/'
n_demos=54 # 12 or 36 or 54 or 128

In [5]:
# Define plotting params
debug_mode = False
c_color, i_color, z_color = 'tab:blue', 'tab:orange', 'tab:green'
first_exit=15 # for risk control, this sets the earliest layer at which we are allowed to exit
plot_directory = './plots/no_calibration/n_demos_' + str(n_demos) + '/'

In [6]:
# Define lambdas and epsilons
# NOTE: If running with loss_01_conversion=max_0, cannot have epsilon < 0!
stepsize = 0.01
eps_grid = np.arange(-0.5, 0.5 + stepsize, stepsize)
lambdas = np.arange(0.0, 1.0 + stepsize, stepsize)[::-1]

In [7]:
# Define loss and risk-control parameters
relative_labels = 'zeroshot_full_model' # zeroshot_full_model or full_model; the predictions over which to compute a relative loss
ground_truth_type = 'true_label' # the ground-truth for computing loss; true_label or zeroshot_full_model
rcp_type = 'ltt'
confidence_type = 'argmax' # argmax or top2_diff or entropy
loss_01_conversion = 'scaling' # None or max_0 or scaling
delta=0.1

# when the risk cannot be controlled for any lambda this is the default loss and exit
# define this as the relative loss
default_loss_uncontrolled_risk = 0 
default_eff_gain_uncontrolled_risk = 0

In [8]:
if debug_mode:
    # Select just a subset of the datasets and models
    # datasets = ['financial_phrasebank']
    n_trials=10
else:
    # Prevent displaying figures
    matplotlib.use('Agg')
    n_trials=25

In [9]:
# Plot accuracy vs layer
eps_grid = np.arange(0.0, 0.5 + stepsize, stepsize)
for dataset in datasets:
    fig, ax = plt.subplots(1,len(models),figsize=(5*len(models),5))
    fig.suptitle(dataset)
    
    for model_idx in range(len(models)):
        model_name, n_early_exit, tokenizer = models[model_idx], n_early_exits[model_idx], tokenizers[model_idx]
        model_ax = ax[model_idx]
        model_ax.set_title(model_name)
        label_order = get_label_order(dataset, tokenizer)
        base_dir = results_folder + '/n_demos_' + str(n_demos) + '/' + dataset + '/' + model_name + '/'
        
        # First check that there exists all types of experiments
        if (os.path.exists(base_dir + 'correct.json') and os.path.exists(base_dir + 'incorrect.json')
                and os.path.exists(base_dir + 'zeroshot.json')):
            # We have all the data
            with open(base_dir + 'correct.json', 'r') as file:
                correct = json.load(file)
            with open(base_dir + 'incorrect.json', 'r') as file:
                incorrect = json.load(file)
            with open(base_dir + 'zeroshot.json', 'r') as file:
                zeroshot = json.load(file)
            c_gt, i_gt, z_gt = get_ground_truth_by_type(ground_truth_type, correct, incorrect, zeroshot, n_early_exit)
            
            for data, gt, label, color in zip([correct, incorrect, zeroshot], [c_gt, i_gt, z_gt], 
                                            ['correct', 'incorrect', 'zeroshot'], [c_color, i_color, z_color]):
                acc = get_all_accuracies(data, gt, n_early_exit, 0)
                model_ax.plot([i for i in range(n_early_exit)], acc.mean(axis=0), label=label, color=color)

            model_ax.legend()
            model_ax.set_xlabel('Layer')
            model_ax.set_ylabel('Accuracy')
        else:
            print('Missing data: ', dataset, model_name)

    plt.tight_layout()
    if debug_mode:
        # Display the image
        plt.show()
    else:
        # Save out the image
        path = plot_directory + 'loss_vs_layer/confidence_' + confidence_type + '/' 
        if not os.path.exists(path):
            os.makedirs(path)
        plt.savefig(path + dataset + '.png')

        # Close plots to save memory
        matplotlib.pyplot.close()
    
    print('Finished', dataset)

Finished financial_phrasebank
Missing data:  sst2 meta-llama/Llama-2-7B-hf
Missing data:  sst2 meta-llama/Llama-2-13B-hf
Finished sst2
Missing data:  tweeteval_hate facebook/layerskip-llama2-7B
Missing data:  tweeteval_hate meta-llama/Meta-Llama-3-8B
Missing data:  tweeteval_hate meta-llama/Llama-2-7B-hf
Missing data:  tweeteval_hate meta-llama/Llama-2-13B-hf
Finished tweeteval_hate
Missing data:  tweeteval_feminist meta-llama/Llama-2-13B-hf
Finished tweeteval_feminist
Missing data:  tweeteval_atheism meta-llama/Llama-2-13B-hf
Finished tweeteval_atheism
Finished unnatural
Finished boolean
Missing data:  navigation meta-llama/Llama-2-13B-hf
Finished navigation
Finished sports
Missing data:  web_of_lies meta-llama/Llama-2-13B-hf
Finished web_of_lies


In [10]:
def plot_single_condition(lambdas, eps_grid, losses, test_risk, eff_gains, rcp_lams, label, color, linestyle):
    # Create plot of lambda vs risk
    ax[0].plot(lambdas, losses.mean(axis=1), label=label, color=color, linestyle=linestyle)
    # Add error bars
    err = np.std(losses, axis=1) / np.sqrt(losses.shape[1])
    ax[0].fill_between(lambdas, losses.mean(axis=1) - err, losses.mean(axis=1) + err, alpha=0.2, color=color)
    ax[0].set_xlabel('Lambda')
    ax[0].set_ylabel('Empirical Risk')
    
    # Create plot of epsilon vs test risk and vs exit layer for optimal lambda
    risk_mean, risk_err = test_risk.mean(axis=0), test_risk.std(axis=0) / np.sqrt(test_risk.shape[0])
    exit_mean, exit_err = n_early_exit - eff_gains.mean(axis=0), eff_gains.std(axis=0) / np.sqrt(eff_gains.shape[0])
    
    ax[1].plot(eps_grid, risk_mean, label=label, color=color, linestyle=linestyle)
    ax[1].fill_between(eps_grid, risk_mean - risk_err, risk_mean + risk_err, alpha=0.2, color=color)
    ax[2].plot(eps_grid, exit_mean, label=label, color=color, linestyle=linestyle)
    ax[2].fill_between(eps_grid, exit_mean - exit_err, exit_mean + exit_err, alpha=0.2, color=color)
    # add a diagonal line and axis labels
    ax[1].plot([min(eps_grid), max(eps_grid)], [min(eps_grid), max(eps_grid)], 'k--')
    ax[1].set_ylabel('Empirical Risk')
    ax[1].set_xlabel('epsilon')
    ax[2].set_ylabel('Average Exit Layer')
    ax[2].set_xlabel('epsilon')

In [11]:
# Plot the two risk-control methods side by side
# For these comparison plots, we need positive epsilon. 
eps_grid = np.arange(0.0, 0.5 + stepsize, stepsize)
for dataset in datasets:
    for model_name, n_early_exit, tokenizer in zip(models, n_early_exits, tokenizers):
        label_order = get_label_order(dataset, tokenizer)
        
        base_dir = results_folder + '/n_demos_' + str(n_demos) + '/' + dataset + '/' + model_name + '/'
        # First check that there exists all types of experiments
        if (os.path.exists(base_dir + 'correct.json') and os.path.exists(base_dir + 'incorrect.json')
                and os.path.exists(base_dir + 'zeroshot.json')):
            # We have all the data
            with open(base_dir + 'correct.json', 'r') as file:
                correct = json.load(file)
            with open(base_dir + 'incorrect.json', 'r') as file:
                incorrect = json.load(file)
            with open(base_dir + 'zeroshot.json', 'r') as file:
                zeroshot = json.load(file)
                
            # We have all the data
            c_gt, i_gt, z_gt = get_ground_truth_by_type(ground_truth_type, correct, incorrect, zeroshot, n_early_exit)
            c_rel, i_rel, z_rel = get_relative_labels(relative_labels, correct, incorrect, zeroshot, n_early_exit)

            fig, ax = plt.subplots(1,3,figsize=(15,5))
            fig.suptitle(dataset + " " + model_name)
            
            for data, gt, rel, label, color in zip([correct, incorrect, zeroshot], [c_gt, i_gt, z_gt], [c_rel, i_rel, z_rel], 
                                            ['correct', 'incorrect', 'zeroshot'], [c_color, i_color, z_color]):
                conf = get_all_confidences(data, n_early_exit, label_order, confidence_type, first_exit)
                acc = get_all_accuracies(data, gt, n_early_exit, first_exit)
                n_cal = int(len(data['0'])/2)
                # Run the max-0 method
                max0_eps_grid = [x for x in eps_grid if x >= 0]
                losses, test_risk, eff_gains, rcp_lams = apply_risk_control(conf, acc, gt, rel, lambdas, max0_eps_grid, rcp_type, 
                                                                            delta, n_cal, n_trials, default_loss_uncontrolled_risk, 
                                                                            default_eff_gain_uncontrolled_risk, 'max_0')
                plot_single_condition(lambdas, max0_eps_grid, losses, test_risk, eff_gains, rcp_lams, label, color, 'dashed')
                # Run the scaling method
                losses, test_risk, eff_gains, rcp_lams = apply_risk_control(conf, acc, gt, rel, lambdas, eps_grid, rcp_type, 
                                                                            delta, n_cal, n_trials, default_loss_uncontrolled_risk, 
                                                                            default_eff_gain_uncontrolled_risk, 'scaling')
                plot_single_condition(lambdas, eps_grid, losses, test_risk, eff_gains, rcp_lams, label + ' with scaling', color, 'solid')

            for a in ax:
                a.legend()
            plt.tight_layout()
            
            if debug_mode:
                # Display the image
                plt.show()
            else:
                # Save out the image
                path = plot_directory + 'risk_control/confidence_' + confidence_type + '/' + dataset + '/'
                if not os.path.exists(path):
                    os.makedirs(path)
                plt.savefig(path + model_name.split('/')[1] + '.png')
    
                # Close plots to save memory
                matplotlib.pyplot.close()
        else:
            print('Missing data: ', dataset, model_name)
    
    print('Finished', dataset)

Finished financial_phrasebank
Missing data:  sst2 meta-llama/Llama-2-7B-hf
Missing data:  sst2 meta-llama/Llama-2-13B-hf
Finished sst2
Missing data:  tweeteval_hate facebook/layerskip-llama2-7B
Missing data:  tweeteval_hate meta-llama/Meta-Llama-3-8B
Missing data:  tweeteval_hate meta-llama/Llama-2-7B-hf
Missing data:  tweeteval_hate meta-llama/Llama-2-13B-hf
Finished tweeteval_hate


KeyboardInterrupt: 

In [ ]:
# Plot confidence of each layer's prediction through layers
# for dataset in datasets:
#     fig, ax = plt.subplots(1,len(models),figsize=(5*len(models),5))
#     plt.suptitle(dataset)
    
#     for model_idx in range(len(models)):
#         model_name, n_early_exit, tokenizer = models[model_idx], n_early_exits[model_idx], tokenizers[model_idx]
#         layers = np.arange(n_early_exit)
#         label_order = get_label_order(dataset, tokenizer)
        
#         data_path = results_folder + dataset + '/' + str(n_demos) + '/' + model_name + '/'
#         if os.path.exists(data_path + 'zeroshot.csv'):
#             # Load data
#             correct = pd.read_csv(data_path + 'correct.csv', engine='python', on_bad_lines='warn').dropna()
#             incorrect = pd.read_csv(data_path + 'incorrect.csv', engine='python', on_bad_lines='warn').dropna()
#             zeroshot = pd.read_csv(data_path + 'zeroshot.csv', engine='python', on_bad_lines='warn').dropna()

#             c_conf = get_all_confidences(correct, n_early_exit, label_order, confidence_type)
#             i_conf = get_all_confidences(incorrect, n_early_exit, label_order, confidence_type)
#             z_conf = get_all_confidences(zeroshot, n_early_exit, label_order, confidence_type)

#             ax[model_idx].plot(layers, c_conf.mean(axis=0), label='correct')
#             ax[model_idx].plot(layers, i_conf.mean(axis=0), label='incorrect')
#             ax[model_idx].plot(layers, z_conf.mean(axis=0), label='zeroshot')
#             ax[model_idx].set_xlabel('Layer')
#             ax[model_idx].set_ylabel('Confidence')
#             ax[model_idx].set_title(model_name)

#     if debug_mode:
#         # Display the image
#         plt.show()
#     else:
#         # Save out the image
#         path = plot_directory + 'confidences/' + confidence_type + '/'
#         if not os.path.exists(path):
#             os.makedirs(path)
#         plt.savefig(path + dataset + '.png')

#     # Close plots to save memory
#     matplotlib.pyplot.close()
#     print('Finished ', dataset)